In [12]:
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.optimize import minimize
import time

df = pd.read_csv('pima_preprocessed_低维.csv')

y = df['y'].values
X = df.iloc[:, 1:].values  
N, d = X.shape

# 模型先验 beta ~ N(0, 10I)
prior_var = 10.0

def log_prior(beta):
    """
    6.4 对数先验
    输出: log_prior (beta) -> 标量对数先验密度
    """
    const_term = -0.5 * d * np.log(2 * np.pi * prior_var)
    quad_term = -0.5 * np.sum(beta**2) / prior_var
    return const_term + quad_term


def log_likelihood(beta):
    """
    6.3 对数似然
    输出: log_likelihood (beta) -> 标量对数似然值
    """
    z = X @ beta
    return np.dot(y, z) - np.sum(np.logaddexp(0, z))


def log_posterior(beta):
    """
    6.5 对数后验
    输出: log_posterior (beta) -> 标量对数后验密度
    """
    return log_likelihood(beta) + log_prior(beta)


def grad_log_posterior(beta):
    """
    6.6 后验梯度
    输出: grad_log_posterior (beta) -> d维梯度向量
    """
    z = X @ beta
    p = expit(z)  # 计算1 / (1 + exp(-z))
    
    grad_ll = X.T @ (y - p)       
    grad_prior = -beta / prior_var  
    return grad_ll + grad_prior


def hmc_sampler(num_samples=10000, burn_in=2000, epsilon=0.10, L=10):
    """
    6.7 后验采样 
    输出: posterior_samples -> 形状为 (num_samples, d) 的采样矩阵
    """
    res = minimize(lambda b: -log_posterior(b), np.zeros(d), jac=lambda b: -grad_log_posterior(b))
    q = res.x.copy()
    
    samples = np.zeros((num_samples, d))
    accepted = 0
    
    for i in range(num_samples + burn_in):
        q_old = q.copy()
        p = np.random.normal(0, 1, size=d)
        p_old = p.copy()
        
        p = p + 0.5 * epsilon * grad_log_posterior(q)
        for l in range(L - 1):
            q = q + epsilon * p
            p = p + epsilon * grad_log_posterior(q)
        q = q + epsilon * p
        p = p + 0.5 * epsilon * grad_log_posterior(q)
        
        current_U = -log_posterior(q_old)
        current_K = 0.5 * np.sum(p_old**2)
        proposed_U = -log_posterior(q)
        proposed_K = 0.5 * np.sum(p**2)
        
        if np.log(np.random.rand()) < (current_U + current_K - proposed_U - proposed_K):
            if i >= burn_in:
                accepted += 1
        else:
            q = q_old
        if i >= burn_in:
            samples[i - burn_in] = q
    acc_rate = accepted / num_samples
    return samples, acc_rate

def compute_ess(samples):
    n = len(samples)
    ess = np.zeros(samples.shape[1])
    for j in range(samples.shape[1]):
        col = samples[:, j]
        mean = np.mean(col)
        var = np.var(col, ddof=1)
        r = np.correlate(col - mean, col - mean, mode='full') / (var * (n - 1))
        r = r[n-1:]
        sum_rho = 0
        for k in range(1, len(r) - 1):
            if r[k] + r[k+1] < 0:
                break
            sum_rho += r[k]
        ess[j] = n / (1 + 2 * sum_rho)
    return ess

if __name__ == "__main__":
    res_map = minimize(lambda b: -log_posterior(b), np.zeros(d), jac=lambda b: -grad_log_posterior(b))
    beta_map = res_map.x
    
    print(f"输入参数beta (MAP 估计值):\n {np.round(beta_map, 4)}")
    print(f"log_likelihood(beta) : {log_likelihood(beta_map):.4f}")
    print(f"log_prior(beta)      : {log_prior(beta_map):.4f}")
    print(f"log_posterior(beta)    : {log_posterior(beta_map):.4f}")
    print(f"grad_log_posterior(beta):\n {np.round(grad_log_posterior(beta_map), 7)}\n")
    
    start_time = time.time()
    posterior_samples, acc_rate = hmc_sampler(num_samples=10000, burn_in=2000)
    runtime = time.time() - start_time
    
    ess_values = compute_ess(posterior_samples)

    print(f"HMC采样接受率: {acc_rate:.2%}")
    print(f"计算效率Runtime: {runtime:.2f} 秒")
    print(f"采样质量ESS:\n {np.round(ess_values, 1)}")
    print(f"平均有效样本数Mean ESS: {np.mean(ess_values):.1f}\n")
    
    feature_names = df.columns[1:]
    samples_df = pd.DataFrame(posterior_samples, columns=feature_names)
    samples_df.to_csv('pima_posterior_samples.csv', index=False)

输入参数beta (MAP 估计值):
 [-0.8602  0.4197  1.1503 -0.1127  0.0308 -0.1005  0.6467  0.2898  0.1533]
log_likelihood(beta) : -356.4212
log_prior(beta)      : -18.7715
log_posterior(beta)    : -375.1927
grad_log_posterior(beta):
 [-2.e-07 -4.e-07 -1.e-07 -3.e-07 -6.e-07  1.e-07 -6.e-07 -2.e-07  1.e-07]

HMC采样接受率: 69.30%
计算效率Runtime: 9.43 秒
采样质量ESS:
 [3543.5 2439.5  343.4  481.6  242.7  162.7  231.9 8646.5 1231.9]
平均有效样本数Mean ESS: 1924.9

